In [0]:
from pyspark.sql import functions as F

app = spark.table("workspace.credit_risk.bronze_application_train")
print(f"Filas: {app.count():,} | Columnas: {len(app.columns)}")
print(f"Tasa de default: {app.agg(F.avg('TARGET')).first()[0]:.2%}")

In [0]:
# Las 15 columnas con más nulos
nulos = app.select([F.avg(F.col(c).isNull().cast("int")).alias(c) for c in app.columns]).first().asDict()
for c, p in sorted(nulos.items(), key=lambda x: -x[1])[:15]:
    print(f"{c:35s} {p:.1%}")

In [0]:
(app.withColumn("sin_vivienda", F.col("COMMONAREA_AVG").isNull().cast("int"))
    .groupBy("sin_vivienda")
    .agg(F.count("*").alias("n"), F.avg("TARGET").alias("tasa_default"))
    .show())

In [0]:
# Valor centinela en DAYS_EMPLOYED: cuál es el valor más frecuente
display(app.groupBy("DAYS_EMPLOYED").count().orderBy(F.desc("count")).limit(5))

In [0]:
# Tasa de default por segmento
for col in ["NAME_CONTRACT_TYPE", "NAME_EDUCATION_TYPE", "NAME_INCOME_TYPE"]:
    display(app.groupBy(col)
               .agg(F.count("*").alias("n"), F.avg("TARGET").alias("tasa_default"))
               .orderBy(F.desc("tasa_default")))

In [0]:
cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
app.agg(*[F.corr(c, "TARGET").alias(c) for c in cols]).show()

# verificación cruzada (pandas excluye nulos por pares)
print(app.select(*cols, "TARGET").toPandas().corr()["TARGET"])

# porcentaje de nulos de cada puntaje externo
app.agg(*[F.avg(F.col(c).isNull().cast("int")).alias(c) for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]).show()

%md
## Hallazgos de la exploración

**1. Desbalance de clases.** 24,825 defaults de 307,511 solicitudes (8.07%).
*Decisión:* no usar accuracy (predecir "todos pagan" acierta ~92%); evaluar con Gini/AUC y KS; partición estratificada por `TARGET`.

**2. Los puntajes externos son las señales individuales más fuertes.** Correlación con `TARGET`: EXT_SOURCE_1 = -0.155, EXT_SOURCE_2 = -0.160, EXT_SOURCE_3 = -0.179 (a mayor puntaje, menor riesgo). Calculadas excluyendo nulos por pares y verificadas con Spark y pandas.
*Decisión:* conservarlas como variables clave. Tienen nulos (sobre todo EXT_SOURCE_1): LightGBM los maneja de forma nativa y en el scorecard los nulos serán un tramo propio ("Missing").

**3. Valor centinela en `DAYS_EMPLOYED`.** 55,374 registros con 365243, frente a 55,362 clientes `Pensioner` (solo 12 de diferencia): el valor codifica "sin empleo actual", no un dato real.
*Decisión:* reemplazar por NULL en silver, crear `EMPLOYED_YEARS = -DAYS_EMPLOYED / 365.25` y conservar la condición de pensionado en `NAME_INCOME_TYPE`.

**4. La ausencia de datos de vivienda es informativa.** Cerca del 70% de los registros no tiene variables de edificio (COMMONAREA, FLOORSMIN, etc.: 69.9%, 69.4%, 68.4%), en bloques `_AVG/_MODE/_MEDI` con el mismo patrón. Tasa de default sin dato: 8.57% (n = 214,865); con dato: 6.91% (n = 92,646); diferencia de 1.66 pp.
*Decisión:* no tratarlo como error; tramo "Missing" en el scorecard y nulos nativos en LightGBM. Es una asociación moderada, no una causa.

**5. El riesgo se concentra en ciertos segmentos (asociaciones, no causas).**
- Educación: el default baja de 10.93% (`Lower secondary`) a 5.36% (`Higher education`). `Academic degree` muestra 1.83% pero con muy pocos casos.
- Producto: los `Cash loans` son el 90.5% de la cartera y tienen 8.35% de default, casi 3 pp más que los `Revolving loans` (5.48%).
- Ingreso: `Working` (158,774 clientes) tiene 9.59% de default frente al 8.07% general: concentra unos 15.2 mil defaults, cerca del 61% del total. `Maternity leave` (40%, 5 casos) y `Unemployed` (36.4%, 22 casos) tienen tasas altísimas pero no son concluyentes.
*Decisión:* reportar siempre el `n` junto a cada tasa y no sacar conclusiones de grupos pequeños.

**Nota de método.** Esta exploración es descriptiva y se hizo sobre todo `application_train`; ningún ajuste aprendido (tramos, imputaciones, codificaciones) se calcula aquí, sino solo con la partición de entrenamiento.